### 1. Persiapan Lingkungan (Setup Environment)

In [66]:
# Install semua library dari file requirements
!pip install -r requirements.txt --quiet
print("✅ Instalasi berhasil!")

✅ Instalasi berhasil!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
import duckdb

# Cek versi untuk memastikan instalasi berhasil
print("--- Check Versions ---")
print(f"DuckDB version: {duckdb.__version__}")
print("----------------------")

--- Check Versions ---
DuckDB version: 1.4.4
----------------------


### 2. Inisialisasi Proyek dbt

In [68]:
# Membuat folder-folder utama dbt untuk memisahkan lapisan transformasi (Layering)
import os

# Daftar folder standar dbt
folders = [
    'models/staging', 
    'models/dimensions', 
    'models/marts', 
    'seeds', 
    'tests', 
    'macros', 
    'analyses', 
    'snapshots'
]

for folder in folders:
    # exist_ok=True itu fungsinya sama kayak -p (nggak error kalau folder udah ada)
    os.makedirs(folder, exist_ok=True)
    print(f"Directory created: {folder}")
print("✅ Struktur direktori dbt berhasil dibuat.")

Directory created: models/staging
Directory created: models/dimensions
Directory created: models/marts
Directory created: seeds
Directory created: tests
Directory created: macros
Directory created: analyses
Directory created: snapshots
✅ Struktur direktori dbt berhasil dibuat.


In [69]:
# Menjalankan skrip ingesti data untuk memuat dataset CSV ke dalam DuckDB
!python "load_source.py"

print("✅ Data ingesti selesai. Database 'classicmodels.duckdb' siap digunakan.")

Loading customers...
Loading employees...
Loading offices...
Loading orders...
Loading payments...
Loading products...
Loading productlines...
Loading orderdetails...
✅ Semua tabel berhasil masuk ke DuckDB!
✅ Data ingesti selesai. Database 'classicmodels.duckdb' siap digunakan.


In [70]:
%%writefile profiles.yml
# Konfigurasi profil koneksi dbt untuk adapter DuckDB
classicmodels_dw:
  outputs:
    dev:
      type: duckdb
      path: 'classicmodels.duckdb'  # Menghubungkan dbt ke file database lokal
      threads: 1                    # Jumlah thread eksekusi paralel
  target: dev

Overwriting profiles.yml


In [71]:
%%writefile dbt_project.yml
# Konfigurasi utama proyek dbt Classic Models
name: 'classicmodels_dw'
version: '1.0.0'
config-version: 2

# Menghubungkan proyek dengan profil koneksi yang telah dibuat sebelumnya
profile: 'classicmodels_dw'

# Mendefinisikan jalur folder untuk komponen dbt
model-paths: ["models"]
seed-paths: ["seeds"]
test-paths: ["tests"]
analysis-paths: ["analyses"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

# Konfigurasi Materialisasi per Layer
models:
  classicmodels_dw:
    staging:
      +materialized: view    # Abstraksi awal dari source data
    dimensions:
      +materialized: table   # Master data yang dioptimalkan
    marts:
      +materialized: table   # Tabel fakta final untuk kebutuhan analitik

Overwriting dbt_project.yml


In [72]:
%%writefile models/sources.yml
version: 2

sources:
  - name: raw_data
    schema: main
    description: "Kumpulan data mentah (raw) dari sistem sumber Classic Models"
    tables:
      - name: customers
        description: "Data profil pelanggan"
      - name: orders
        description: "Informasi transaksi pemesanan"
      - name: orderdetails
        description: "Detail item per pesanan"
      - name: products
        description: "Katalog produk perusahaan"
      - name: employees
        description: "Data internal karyawan"
      - name: offices
        description: "Informasi lokasi kantor cabang"
      - name: payments
        description: "Riwayat transaksi pembayaran pelanggan"
      - name: productlines
        description: "Kategori dan deskripsi lini produk"

Overwriting models/sources.yml


In [73]:
%%writefile models/marts/schema.yml
version: 2

models:
  - name: fct_payments
    description: "Tabel fakta yang mencatat setiap transaksi pembayaran dari pelanggan."
    columns:
      - name: check_number
        description: "ID unik transaksi pembayaran"
        tests:
          - unique
          - not_null

      - name: customer_key
        description: "Kunci asing yang menghubungkan ke tabel dim_customer"
        tests:
          - not_null
          - relationships:
              to: ref('dim_customer')
              field: customer_key

      - name: payment_date_id
        description: "Kunci asing yang menghubungkan ke tabel dim_date"
        tests:
          - not_null
          - relationships:
              to: ref('dim_date')
              field: date_id

      - name: amount
        description: "Nilai nominal pembayaran"
        tests:
          - not_null

Overwriting models/marts/schema.yml


In [74]:
# Menjalankan uji validasi sistem dan koneksi database
!dbt debug --profiles-dir .

04:52:54  Running with dbt=1.10.20
04:52:54  dbt version: 1.10.20
04:52:54  python version: 3.9.16
04:52:54  python path: C:\Users\ASUS\miniconda3\python.exe
04:52:54  os info: Windows-10-10.0.19045-SP0
04:52:55  Using profiles dir at .
04:52:55  Using profiles.yml file at .\profiles.yml
04:52:55  Using dbt_project.yml file at f:\S2\dw\classicmodels-dbt-duckdb\dbt_project.yml
04:52:55  adapter type: duckdb
04:52:55  adapter version: 1.10.0
04:52:55  Configuration:
04:52:55    profiles.yml file [OK found and valid]
04:52:55    dbt_project.yml file [OK found and valid]
04:52:55  Required dependencies:
04:52:55   - git [OK found]

04:52:55  Connection:
04:52:55    database: classicmodels
04:52:55    schema: main
04:52:55    path: classicmodels.duckdb
04:52:55    config_options: None
04:52:55    extensions: None
04:52:55    settings: {}
04:52:55    external_root: .
04:52:55    use_credential_provider: None
04:52:55    attach: None
04:52:55    filesystems: None
04:52:55    remote: None
04:5

### 3. Pengembangan Data Marts dengan dbt

##### Langkah 1: Verifikasi Data Sumber (Source Data Verification)

In [75]:
import duckdb
import pandas as pd

# Path ke database DuckDB
db_path = 'classicmodels.duckdb'

# Step 1: Memverifikasi daftar tabel yang tersedia di database
print("--- Daftar Tabel dalam Database ---")
with duckdb.connect(db_path) as con:
    df_tables = con.execute("""
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = 'main'
        ORDER BY table_name;
    """).df()
    display(df_tables)

# Step 2: Melakukan preview relasi antar tabel (Order Chain)
print("\n--- Preview Relasi Orders & Customers ---")
query_preview = """
SELECT o.orderNumber,
       o.orderDate,
       o.status,
       c.customerName,
       c.country
FROM orders o
JOIN customers c USING (customerNumber)
LIMIT 10;
"""

with duckdb.connect(db_path) as con:
    df_preview = con.execute(query_preview).df()
    display(df_preview)

--- Daftar Tabel dalam Database ---


,table_name,table_type
0,customers,BASE TABLE
1,dim_customer,BASE TABLE
2,dim_date,BASE TABLE
3,dim_employee,BASE TABLE
4,dim_office,BASE TABLE
5,dim_product,BASE TABLE
6,employees,BASE TABLE
7,fct_order_lines,BASE TABLE
8,fct_payments,BASE TABLE
9,fct_sales_performance,BASE TABLE



--- Preview Relasi Orders & Customers ---


,orderNumber,orderDate,status,customerName,country
0,10100,2003-01-06,Shipped,Online Diecast Creations Co.,USA
1,10101,2003-01-09,Shipped,"Blauer See Auto, Co.",Germany
2,10102,2003-01-10,Shipped,Vitachrome Inc.,USA
3,10103,2003-01-29,Shipped,Baane Mini Imports,Norway
4,10104,2003-01-31,Shipped,Euro+ Shopping Channel,Spain
5,10105,2003-02-11,Shipped,Danish Wholesale Imports,Denmark
6,10106,2003-02-17,Shipped,Rovelli Gifts,Italy
7,10107,2003-02-24,Shipped,Land of Toys Inc.,USA
8,10108,2003-03-03,Shipped,Cruz & Sons Co.,Philippines
9,10109,2003-03-10,Shipped,Motor Mint Distributors Inc.,USA


##### Langkah 2: Membangun Staging Layer

In [76]:
%%writefile models/staging/stg_orders.sql
select
    orderNumber    as order_id,
    orderDate      as order_date,
    requiredDate   as required_date,
    shippedDate    as shipped_date,
    status         as order_status,
    comments       as comments,
    customerNumber as customer_id
from {{ source('raw_data', 'orders') }}

Overwriting models/staging/stg_orders.sql


In [77]:
%%writefile models/staging/stg_orderdetails.sql
select
    orderNumber      as order_id,
    productCode      as product_code,
    quantityOrdered  as quantity_ordered,
    priceEach        as price_each,
    orderLineNumber  as order_line_number
from {{ source('raw_data', 'orderdetails') }}

Overwriting models/staging/stg_orderdetails.sql


In [78]:
%%writefile models/staging/stg_customers.sql
SELECT
    customerNumber AS customer_id,
    customerName AS customer_name,
    contactLastName AS contact_last_name,
    contactFirstName AS contact_first_name,
    phone,
    salesRepEmployeeNumber AS sales_rep_employee_id,
    city,
    country
FROM {{ source('raw_data', 'customers') }}

Overwriting models/staging/stg_customers.sql


In [79]:
%%writefile models/staging/stg_products.sql
SELECT
    productCode AS product_code,
    productName AS product_name,
    productLine AS product_line,
    productScale AS product_scale,
    productVendor AS product_vendor,
    quantityInStock AS quantity_in_stock,
    buyPrice AS buy_price,
    MSRP AS msrp
FROM {{ source('raw_data', 'products') }}

Overwriting models/staging/stg_products.sql


In [80]:
%%writefile models/staging/stg_employees.sql
SELECT
    employeeNumber AS employee_id,
    lastName AS last_name,
    firstName AS first_name,
    extension,
    email,
    officeCode AS office_code,
    reportsTo AS reports_to_employee_id,
    jobTitle AS job_title
FROM {{ source('raw_data', 'employees') }}

Overwriting models/staging/stg_employees.sql


In [81]:
%%writefile models/staging/stg_offices.sql
SELECT
    officeCode AS office_code,
    city,
    phone,
    addressLine1 AS address_line_1,
    country,
    territory
FROM {{ source('raw_data', 'offices') }}

Overwriting models/staging/stg_offices.sql


In [82]:
%%writefile models/staging/stg_payments.sql
SELECT
    customerNumber AS customer_id,
    checkNumber AS check_number,
    paymentDate AS payment_date,
    amount
FROM {{ source('raw_data', 'payments') }}

Overwriting models/staging/stg_payments.sql


In [83]:
%%writefile models/staging/stg_productlines.sql
SELECT
    productLine AS product_line,
    textDescription AS text_description
FROM {{ source('raw_data', 'productlines') }}

Overwriting models/staging/stg_productlines.sql


In [84]:
# run dbt staging
!dbt run --profiles-dir . --select staging

04:53:09  Running with dbt=1.10.20
04:53:10  Registered adapter: duckdb=1.10.0
04:53:13  Found 16 models, 7 data tests, 8 sources, 456 macros
04:53:13  
04:53:13  Concurrency: 1 threads (target='dev')
04:53:13  
04:53:14  1 of 8 START sql view model main.stg_customers ................................. [RUN]
04:53:14  1 of 8 OK created sql view model main.stg_customers ............................ [OK in 0.32s]
04:53:14  2 of 8 START sql view model main.stg_employees ................................. [RUN]
04:53:14  2 of 8 OK created sql view model main.stg_employees ............................ [OK in 0.29s]
04:53:14  3 of 8 START sql view model main.stg_offices ................................... [RUN]
04:53:15  3 of 8 OK created sql view model main.stg_offices .............................. [OK in 0.25s]
04:53:15  4 of 8 START sql view model main.stg_orderdetails .............................. [RUN]
04:53:15  4 of 8 OK created sql view model main.stg_orderdetails ....................

In [85]:
# checkpoint
query = """SELECT order_id, order_date, order_status, customer_id
FROM stg_orders
LIMIT  5;"""

with duckdb.connect(db_path) as con:
    # Cek apakah view stg_orders sudah muncul
    df = con.execute(query).df()
    display(df)

query = """SELECT order_id, product_code, quantity_ordered, price_each
FROM stg_orderdetails
LIMIT 5;"""

with duckdb.connect(db_path) as con:
    # Cek apakah view stg_orders sudah muncul
    df = con.execute(query).df()
    display(df)


,order_id,order_date,order_status,customer_id
0,10100,2003-01-06,Shipped,363
1,10101,2003-01-09,Shipped,128
2,10102,2003-01-10,Shipped,181
3,10103,2003-01-29,Shipped,121
4,10104,2003-01-31,Shipped,141


,order_id,product_code,quantity_ordered,price_each
0,10100,S18_1749,30,136.00
1,10100,S18_2248,50,55.09
2,10100,S18_4409,22,75.46
3,10100,S24_3969,49,35.29
4,10101,S18_2325,25,108.06


##### Langkah 3: Membangun Model Dimensi

In [86]:
%%writefile models/dimensions/dim_date.sql
with date_spine as (
    -- Date range covers all ClassicModels order dates.
    -- Extended to full calendar years for clean reporting.
    select generate_series::date as date_day
    from generate_series(
        date '2003-01-01',
        date '2006-12-31',
        interval '1 day'
    )
)
select
    date_day                                as date_id,
    date_day                                as full_date,
    extract('year'    from date_day)::int   as year,
    extract('quarter' from date_day)::int   as quarter,
    extract('month'   from date_day)::int   as month,
    strftime('%B', date_day)                as month_name,
    extract('week'    from date_day)::int   as week_of_year,
    extract('day'     from date_day)::int   as day_of_month,
    extract('dow'     from date_day)::int   as day_of_week,
    strftime('%A', date_day)                as day_name,
    (extract('dow' from date_day) in (0, 6)) as is_weekend
from date_spine

Overwriting models/dimensions/dim_date.sql


In [87]:
%%writefile models/dimensions/dim_product.sql
select
    row_number() over (order by p.product_code)   as product_key,
    p.product_code,
    p.product_name,
    p.product_line,
    pl.text_description                           as product_line_description,
    p.product_scale,
    p.product_vendor,
    p.buy_price,
    p.msrp
from {{ ref('stg_products') }} p
left join {{ ref('stg_productlines') }} pl
    on p.product_line = pl.product_line

Overwriting models/dimensions/dim_product.sql


In [88]:
%%writefile models/dimensions/dim_employee.sql
select
    row_number() over (order by e.employee_id) as employee_key,
    e.employee_id,
    e.first_name,
    e.last_name,
    e.first_name || ' ' || e.last_name as full_name,
    e.job_title,
    e.email,
    e.office_code,
    e.reports_to_employee_id,
    mgr.first_name || ' ' || mgr.last_name as manager_full_name
from {{ ref('stg_employees') }} e
left join {{ ref('stg_employees') }} mgr
    on e.reports_to_employee_id = mgr.employee_id

Overwriting models/dimensions/dim_employee.sql


In [89]:
%%writefile models/dimensions/dim_customer.sql
SELECT
    row_number() over (order by customer_id) as customer_key,
    customer_id,
    customer_name,
    contact_first_name || ' ' || contact_last_name AS full_contact_name,
    phone,
    sales_rep_employee_id,
    city,
    country
FROM {{ ref('stg_customers') }}



Overwriting models/dimensions/dim_customer.sql


In [90]:
%%writefile models/dimensions/dim_office.sql
SELECT
    row_number() over (order by office_code) as office_key,
    office_code,
    city,
    phone,
    address_line_1,
    country,
    territory
FROM {{ ref('stg_offices') }}

Overwriting models/dimensions/dim_office.sql


In [91]:
!dbt run --profiles-dir . --select dimensions

04:53:27  Running with dbt=1.10.20
04:53:29  Registered adapter: duckdb=1.10.0
04:53:32  Found 16 models, 7 data tests, 8 sources, 456 macros
04:53:32  
04:53:32  Concurrency: 1 threads (target='dev')
04:53:32  
04:53:33  1 of 5 START sql table model main.dim_customer ................................. [RUN]
04:53:33  1 of 5 OK created sql table model main.dim_customer ............................ [OK in 0.52s]
04:53:33  2 of 5 START sql table model main.dim_date ..................................... [RUN]
04:53:34  2 of 5 OK created sql table model main.dim_date ................................ [OK in 0.29s]
04:53:34  3 of 5 START sql table model main.dim_employee ................................. [RUN]
04:53:34  3 of 5 OK created sql table model main.dim_employee ............................ [OK in 0.21s]
04:53:34  4 of 5 START sql table model main.dim_office ................................... [RUN]
04:53:34  4 of 5 OK created sql table model main.dim_office .........................

In [92]:
# 1. Audit Dimensi Waktu
print("--- Verifikasi Rentang Data dim_date ---")
query_date = """
SELECT
    MIN(date_id) AS start_date,
    MAX(date_id) AS end_date,
    COUNT(*) AS total_days
FROM dim_date;
"""

with duckdb.connect(db_path) as con:
    df_date = con.execute(query_date).df()
    display(df_date)
    # Catatan: Pastikan jumlah hari sesuai dengan kalender (termasuk kabisat)

# 2. Audit Hirarki Karyawan
print("\n--- Verifikasi Hirarki Manajer dim_employee ---")
query_emp = """
SELECT
    full_name,
    job_title,
    manager_full_name
FROM dim_employee
ORDER BY employee_id;
"""

with duckdb.connect(db_path) as con:
    df_emp = con.execute(query_emp).df()
    display(df_emp)

--- Verifikasi Rentang Data dim_date ---


,start_date,end_date,total_days
0,2003-01-01,2006-12-31,1461



--- Verifikasi Hirarki Manajer dim_employee ---


,full_name,job_title,manager_full_name
0,Diane Murphy,President,None
1,Mary Patterson,VP Sales,Diane Murphy
2,Jeff Firrelli,VP Marketing,Diane Murphy
3,William Patterson,Sales Manager (APAC),Mary Patterson
4,Gerard Bondur,Sale Manager (EMEA),Mary Patterson
5,Anthony Bow,Sales Manager (NA),Mary Patterson
6,Leslie Jennings,Sales Rep,Anthony Bow
7,Leslie Thompson,Sales Rep,Anthony Bow
8,Julie Firrelli,Sales Rep,Anthony Bow
9,Steve Patterson,Sales Rep,Anthony Bow


##### Langkah 4: Build the Fact Tables

In [93]:
%%writefile models/marts/fct_order_lines.sql
select
    od.order_id,
    od.order_line_number,
    o.order_date                           as order_date_id,
    dc.customer_key,
    dp.product_key,
    de.employee_key                        as sales_rep_key,
    dof.office_key,
    od.quantity_ordered,
    od.price_each,
    round(od.quantity_ordered * od.price_each, 2) as revenue,
    dp.buy_price,
    round(od.quantity_ordered * dp.buy_price, 2) as cost,
    round(
        (od.quantity_ordered * od.price_each)
      - (od.quantity_ordered * dp.buy_price), 2
    ) as gross_profit
from {{ ref('stg_orderdetails') }} od
join {{ ref('stg_orders') }} o
    on od.order_id = o.order_id
join {{ ref('dim_customer') }} dc
    on o.customer_id = dc.customer_id
join {{ ref('dim_product') }} dp
    on od.product_code = dp.product_code
left join {{ ref('dim_employee') }} de
    on dc.sales_rep_employee_id = de.employee_id
left join {{ ref('dim_office') }} dof
    on de.office_code = dof.office_code

Overwriting models/marts/fct_order_lines.sql


In [94]:
%%writefile models/marts/fct_payments.sql
-- depends_on: {{ ref('dim_date') }}
select
    p.check_number,
    p.payment_date as payment_date_id,
    dc.customer_key,
    p.amount
from {{ ref('stg_payments') }} p
join {{ ref('dim_customer') }} dc
    on p.customer_id = dc.customer_id

Overwriting models/marts/fct_payments.sql


In [95]:
%%writefile models/marts/fct_sales_performance.sql
select
    sales_rep_key,
    office_key,
    date_trunc('month', order_date_id)::date     as month_date,
    extract('year'  from order_date_id)::int     as year,
    extract('month' from order_date_id)::int     as month,
    count(distinct order_id)                     as order_count,
    count(distinct customer_key)                 as unique_customers,
    sum(quantity_ordered)                        as total_units_sold,
    round(sum(revenue), 2)                       as total_revenue,
    round(sum(gross_profit), 2)                  as total_gross_profit,
    round(avg(revenue), 2)                       as avg_order_line_revenue
from {{ ref('fct_order_lines') }}
-- Orders with no assigned sales rep are excluded (rep-level analysis only).
where sales_rep_key is not null
group by 1, 2, 3, 4, 5

Overwriting models/marts/fct_sales_performance.sql


In [96]:
# Mengeksekusi seluruh model marts
!dbt run --profiles-dir . --select marts

04:53:44  Running with dbt=1.10.20
04:53:45  Registered adapter: duckdb=1.10.0
04:53:47  Found 16 models, 7 data tests, 8 sources, 456 macros
04:53:47  
04:53:47  Concurrency: 1 threads (target='dev')
04:53:47  
04:53:48  1 of 3 START sql table model main.fct_order_lines .............................. [RUN]
04:53:49  1 of 3 OK created sql table model main.fct_order_lines ......................... [OK in 0.39s]
04:53:49  2 of 3 START sql table model main.fct_payments ................................. [RUN]
04:53:49  2 of 3 OK created sql table model main.fct_payments ............................ [OK in 0.30s]
04:53:49  3 of 3 START sql table model main.fct_sales_performance ........................ [RUN]
04:53:49  3 of 3 OK created sql table model main.fct_sales_performance ................... [OK in 0.20s]
04:53:49  
04:53:49  Finished running 3 table models in 0 hours 0 minutes and 2.07 seconds (2.07s).
04:53:49  
04:53:49  Completed successfully
04:53:49  
04:53:49  Done. PASS=3 WARN

##### Langkah 5: Validasi Kualitas Data (Data Testing)

In [97]:
# Menjalankan pengujian kualitas data khusus untuk tabel fct_payments
!dbt test --select fct_payments --profiles-dir .

print("\n✅ Validasi integritas tabel fct_payments selesai.")

04:53:58  Running with dbt=1.10.20
04:53:59  Registered adapter: duckdb=1.10.0
04:54:01  Found 16 models, 7 data tests, 8 sources, 456 macros
04:54:01  
04:54:01  Concurrency: 1 threads (target='dev')
04:54:01  
04:54:02  1 of 7 START test not_null_fct_payments_amount ................................. [RUN]
04:54:02  1 of 7 PASS not_null_fct_payments_amount ....................................... [PASS in 0.18s]
04:54:02  2 of 7 START test not_null_fct_payments_check_number ........................... [RUN]
04:54:02  2 of 7 PASS not_null_fct_payments_check_number ................................. [PASS in 0.06s]
04:54:02  3 of 7 START test not_null_fct_payments_customer_key ........................... [RUN]
04:54:02  3 of 7 PASS not_null_fct_payments_customer_key ................................. [PASS in 0.07s]
04:54:02  4 of 7 START test not_null_fct_payments_payment_date_id ........................ [RUN]
04:54:02  4 of 7 PASS not_null_fct_payments_payment_date_id ...................

##### Langkah 6: Eksplorasi Data Marts & Analisis Bisnis

###### Skenario 1: Identifikasi Produk "High-Volume, Low-Margin"

In [98]:
query = """
SELECT
    p.product_name,
    SUM(f.quantity_ordered) AS total_units_sold,
    ROUND(SUM(f.revenue), 2) AS total_revenue,
    ROUND(SUM(f.gross_profit), 2) AS total_profit,
    ROUND(SUM(f.gross_profit) / NULLIF(SUM(f.revenue), 0) * 100, 2) AS profit_margin_pct
FROM 'fct_order_lines' f
JOIN 'dim_product' p ON f.product_key = p.product_key
GROUP BY 1
ORDER BY total_units_sold DESC
LIMIT 10;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)

,product_name,total_units_sold,total_revenue,total_profit,profit_margin_pct
0,1992 Ferrari 360 Spider red,1808.0,276839.98,135996.78,49.12
1,1937 Lincoln Berline,1111.0,102563.52,35214.70,34.33
2,American Airlines: MD-11S,1085.0,71753.93,32400.98,45.16
3,1941 Chevrolet Special Deluxe Cabriolet,1076.0,102537.45,33049.37,32.23
4,1930 Buick Marquette Phaeton,1074.0,41599.24,12536.80,30.14
5,1940s Ford truck,1061.0,114232.79,24302.43,21.27
6,1969 Harley Davidson Ultimate Chopper,1057.0,90157.77,38565.60,42.78
7,1957 Chevy Pickup,1056.0,109946.21,51127.01,46.50
8,1964 Mercedes Tour Bus,1053.0,117669.66,38842.08,33.01
9,1956 Porsche 356A Coupe,1052.0,134240.71,30829.11,22.97


###### Skenario 2: Analisis Likuiditas (Sales vs Payment)

In [99]:
# Query ini membandingkan akumulasi pesanan pelanggan dengan realisasi pembayaran yang sudah diterima.

# Identifikasi Produk "High-Volume, Low-Margin"
query = """
WITH customer_sales AS (
    SELECT
        customer_key,
        SUM(revenue) AS total_ordered
    FROM fct_order_lines
    GROUP BY 1
),
customer_payments AS (
    SELECT
        customer_key,
        SUM(amount) AS total_paid
    FROM fct_payments
    GROUP BY 1
)
SELECT
    c.customer_name,
    s.total_ordered,
    COALESCE(p.total_paid, 0) AS total_paid,
    ROUND((s.total_ordered - COALESCE(p.total_paid, 0)),2) AS outstanding_balance
FROM customer_sales s
LEFT JOIN customer_payments p ON s.customer_key = p.customer_key
JOIN dim_customer c ON s.customer_key = c.customer_key
where outstanding_balance > 0
ORDER BY outstanding_balance DESC;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


,customer_name,total_ordered,total_paid,outstanding_balance
0,Euro+ Shopping Channel,820689.54,715738.98,104950.56
1,The Sharp Gifts Warehouse,143536.27,59551.38,83984.89
2,Salzburg Collectables,137480.07,85060.00,52420.07
3,Gifts4AllAges.com,84340.32,33533.47,50806.85
4,"UK Collectables, Ltd.",106610.72,61167.18,45443.54
5,Scandinavian Gift Ideas,120943.53,76776.44,44167.09
6,Tekni Collectables Inc.,81806.55,38281.51,43525.04
7,Souveniers And Things Co.,133907.12,91655.61,42251.51
8,La Rochelle Gifts,158573.12,116949.68,41623.44
9,Land of Toys Inc.,149085.15,107639.94,41445.21


###### Skenario 3: Evaluasi Efisiensi Sales Representative

In [100]:
# Query ini mencari sales yang paling efisien dalam menghasilkan profit per transaksi.
query = """
SELECT
    e.first_name || ' ' || e.last_name AS sales_rep_name,
    SUM(order_count) AS total_orders,
    SUM(total_revenue) AS total_revenue,
    SUM(total_gross_profit) AS total_profit,
    ROUND(SUM(total_gross_profit) / SUM(order_count), 2) AS avg_profit_per_order
FROM fct_sales_performance f
JOIN dim_employee e ON f.sales_rep_key = e.employee_key
GROUP BY 1
ORDER BY avg_profit_per_order DESC;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


,sales_rep_name,total_orders,total_revenue,total_profit,avg_profit_per_order
0,Larry Bott,22.0,732096.79,290203.59,13191.07
1,Martin Gerard,12.0,387477.47,156878.63,13073.22
2,Leslie Jennings,34.0,1081530.54,435208.35,12800.25
3,George Vanauf,22.0,669377.05,269596.09,12254.37
4,Peter Marsh,19.0,584593.76,230811.75,12147.99
5,Loui Bondur,20.0,569485.75,234891.07,11744.55
6,Gerard Hernandez,43.0,1258577.81,504644.71,11735.92
7,Andy Fixter,19.0,562582.59,222207.18,11695.11
8,Foon Yue Tseng,17.0,488212.67,194839.92,11461.17
9,Mami Nishi,16.0,457110.07,181181.80,11323.86


###### Skenario 4: Pemetaan Tren Penjualan Musiman

In [101]:
# Query ini menggunakan tabel dimensi waktu untuk melihat performa per kuartal secara historis.
query = """
SELECT
    d.year,
    d.quarter,
    ROUND(SUM(f.revenue), 2) AS periodic_revenue,
    ROUND(SUM(f.gross_profit), 2) AS periodic_profit
FROM fct_order_lines f
JOIN dim_date d ON f.order_date_id = d.date_id
GROUP BY 1, 2
ORDER BY 1, 2;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


,year,quarter,periodic_revenue,periodic_profit
0,2003,1,405613.55,163904.29
1,2003,2,515754.91,203663.37
2,2003,3,616895.32,246125.00
3,2003,4,1779084.61,706930.28
4,2004,1,799579.31,319596.74
5,2004,2,779271.81,318075.96
6,2004,3,1028690.38,409638.02
7,2004,4,1908364.01,762070.42
8,2005,1,984641.15,388032.15
9,2005,2,786295.56,307844.02


###### Skenario 5: Optimalisasi Operasional Kantor Cabang

In [102]:
# Query ini memantau produktivitas kantor cabang berdasarkan tren bulanan.
query = """
SELECT
    o.city AS office_location,
    f.month_date,
    f.unique_customers,
    f.total_revenue,
    f.total_gross_profit
FROM fct_sales_performance f
JOIN dim_office o ON f.office_key = o.office_key
ORDER BY 1, 2;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


,office_location,month_date,unique_customers,total_revenue,total_gross_profit
0,Boston,2003-01-01,1,10223.83,3940.36
1,Boston,2003-06-01,1,6036.96,1841.42
2,Boston,2003-08-01,1,41016.75,15435.36
3,Boston,2003-09-01,1,32680.31,12452.76
4,Boston,2003-10-01,1,9977.85,4416.09
...,...,...,...,...,...
213,Tokyo,2004-08-01,1,2611.84,967.40
214,Tokyo,2004-11-01,2,64750.48,28271.94
215,Tokyo,2005-01-01,1,33967.73,11554.27
216,Tokyo,2005-03-01,1,3516.04,571.56


##### Langkah 7: Visualisasi Silsilah Data (dbt DAG)

In [103]:
# Menghasilkan metadata dokumentasi proyek
!dbt docs generate --profiles-dir .

04:54:11  Running with dbt=1.10.20
04:54:12  Registered adapter: duckdb=1.10.0
04:54:14  Found 16 models, 7 data tests, 8 sources, 456 macros
04:54:14  
04:54:14  Concurrency: 1 threads (target='dev')
04:54:14  
04:54:15  Building catalog
04:54:16  Catalog written to f:\S2\dw\classicmodels-dbt-duckdb\target\catalog.json


In [104]:
"""
Jika menjalankan secara Lokal (Laptop/PC):
Gunakan perintah standar dbt di terminal untuk membuka server lokal di
http://localhost:8080.
"""

!dbt docs serve --profiles-dir .

04:54:25  Running with dbt=1.10.20
04:54:26  Encountered an error:
[WinError 10048] Only one usage of each socket address (protocol/network address/port) is normally permitted
04:54:26  Traceback (most recent call last):
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\requires.py", line 178, in wrapper
    result, success = func(*args, **kwargs)
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\requires.py", line 128, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\requires.py", line 272, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\requires.py", line 317, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\requires.py", line 364, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\ASUS\miniconda3\lib\site-packages\dbt\cli\main.py", line 307, in docs_serve
    results = task.run()
  File "C:\User

In [105]:
# """
# Jika menjalankan di Google Colab:
# Karena Colab berjalan di server cloud Google,
# kita perlu menggunakan proxy internal agar browser Anda bisa mengakses folder target yang ada di server tersebut.
# """
# from google.colab import output
# import os

# # Pastikan folder target ada
# if os.path.exists("target"):
#     # Jalankan server internal
#     port = 8000
#     get_ipython().system_raw(f'python3 -m http.server {port} --directory target &')

#     # Berikan link proxy internal Google
#     print("Klik link di bawah ini untuk melihat docs:")
#     print(output.eval_js(f'google.colab.kernel.proxyPort({port})'))
# else:
#     print("Folder 'target' tidak ditemukan. Jalankan '!dbt docs generate' dulu!")